# SafeGrid AI: Safety Evacuation Planning using LLMs
### End-to-End Walkthrough on CubiCasa5k Architectural Blueprints

This notebook demonstrates the complete SafeGrid AI pipeline:
1. **Parse CubiCasa5k SVGs into Spatial Polygons**
2. **Construct Topological Navigation Graphs with NetworkX**
3. **Simulate Dynamic Hazards with Exponential Weight Penalization**
4. **Deterministic Dijkstra Routing Tool Execution**
5. **LLM Agent Emergency Coordination (Gemini API / Tool Binding)**
6. **Pipeline Benchmarks & Evaluation**

In [ ]:
import os
import sys
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.svg_parser import SVGFloorPlanParser
from src.graph_builder import GraphBuilder
from src.hazard_engine import HazardEngine
from src.pathfinder import SafePathfinder
from src.llm_agent import EvacuationAgent
import config

print("SafeGrid AI modules successfully imported!")

## 1. Parse CubiCasa5k Blueprint SVG

In [ ]:
svg_path = os.path.join(config.DATASET_ROOT, "high_quality_architectural", "1", "model.svg")
parser = SVGFloorPlanParser()
data = parser.parse(svg_path)

print(f"Extracted {len(data['spaces'])} spaces/rooms")
print(f"Extracted {len(data['doors'])} interior doors")
print(f"Extracted {len(data['exits'])} exterior exits")
print(f"Extracted {len(data['windows'])} windows")

## 2. Build Topological Navigation Graph

In [ ]:
builder = GraphBuilder()
G = builder.build_graph(data)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print("Sample nodes:", list(G.nodes(data=True))[:3])

## 3. Dynamic Hazard Weighting (SafeGrid Logic)
Inject a synthetic hazard (e.g. fire) and apply exponential penalization:

In [ ]:
engine = HazardEngine(G)
hazard_id = engine.add_hazard(x=500.0, y=400.0, radius=150.0, hazard_type="fire", severity=1.2)
print(f"Injected hazard: {hazard_id}")

weighted_G = engine.get_weighted_graph()
print("Weights updated based on proximity to hazard.")

## 4. Deterministic Dijkstra Safest Pathfinding Tool

In [ ]:
pathfinder = SafePathfinder(weighted_G)
route_result = pathfinder.find_safest_route(start_node="Room_0", exit_node="Exit_0")

print("Safest Route:", " -> ".join(route_result["path"]))
print(f"Total Traversal Cost: {route_result['cost']:.2f}")
print(f"Safety Margin: {route_result['safety_margin']:.2f} units from hazard")
print(f"Route Valid along graph: {pathfinder.is_path_valid(route_result['path'])}")

## 5. Agentic LLM Emergency Evacuation Coordinator

In [ ]:
agent = EvacuationAgent()
evac_result = agent.generate_instructions(
    graph=weighted_G,
    start_node="Room_0",
    hazards=engine.get_hazards(),
    exit_node="Exit_0"
)

print(evac_result.instructions)

## 6. Visualizing Benchmark Results

In [ ]:
from IPython.display import Image, display

results_dir = os.path.join(PROJECT_ROOT, "benchmarks", "results")
for img_file in ["routing_benchmark.png", "hazard_proximity_chart.png", "hazard_escalation_chart.png", "llm_vs_agentic.png", "sample_graph.png"]:
    img_path = os.path.join(results_dir, img_file)
    if os.path.exists(img_path):
        print(f"\n--- {img_file} ---")
        display(Image(filename=img_path))